# Chapter 13a — The Perceptron: a Single Artificial Neuron

Welcome to the **capstone** of the course. Everything you have learned —
vectors and dot products (Ch. 03), matrices (Ch. 04), derivatives and the
chain rule (Ch. 06–07), and gradient descent (Ch. 12) — comes together to
build a **neural network from scratch**, using nothing but NumPy.

We start small: a *single neuron*. A neuron takes inputs $x$, forms a
**weighted sum**

$$z = w \cdot x + b = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b,$$

then passes $z$ through a nonlinear **activation function** $a = \sigma(z)$.
That is the entire object. A neural network is just *many* of these wired
together — which is the subject of notebook 13b.

In this notebook we will:

1. build one neuron and the three classic activation functions,
2. train a neuron with the **perceptron learning rule** on a separable
   dataset, and
3. watch it **fail on XOR** — the failure that forces us to invent
   *hidden layers*.

## 1. Setup

Pure NumPy and Matplotlib. We fix the random seed with
`np.random.default_rng(0)` so every run is identical and reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # one reproducible random generator

## 2. The weighted sum $z = w \cdot x + b$

A neuron's input is a vector $x \in \mathbb{R}^n$. It carries a weight vector
$w \in \mathbb{R}^n$ and a single number $b$ (the **bias**). The "pre-activation"
is the dot product plus the bias — exactly the dot product from Chapter 03.

In [ ]:
def pre_activation(w, x, b):
    return np.dot(w, x) + b      # w . x + b  (a single number)

w = np.array([2.0, -1.0])        # weights
b = 0.5                          # bias
x = np.array([1.0, 3.0])         # an input point

print("z =", pre_activation(w, x, b))   # 2*1 + (-1)*3 + 0.5 = -0.5

## 3. Activation functions

The weighted sum $z$ is linear. To let neurons represent *curved* relationships
we squash $z$ through a nonlinear **activation** $\sigma$. The three classics:

$$
\text{sigmoid}(z) = \frac{1}{1+e^{-z}}, \qquad
\tanh(z) = \frac{e^{z}-e^{-z}}{e^{z}+e^{-z}}, \qquad
\text{ReLU}(z) = \max(0, z).
$$

- **sigmoid** squashes into $(0,1)$ — natural for probabilities (Ch. 11).
- **tanh** squashes into $(-1,1)$ — like sigmoid but centred at $0$.
- **ReLU** ("rectified linear unit") is $0$ for negatives, identity for
  positives — cheap and the modern default for hidden layers.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def tanh(z):
    return np.tanh(z)            # NumPy already has tanh

def relu(z):
    return np.maximum(0.0, z)    # elementwise max with 0

Let's plot all three on the same axes so you can see their shapes.

In [ ]:
z = np.linspace(-6, 6, 200)      # 200 points from -6 to 6

plt.figure(figsize=(7, 4))
plt.plot(z, sigmoid(z), label="sigmoid")
plt.plot(z, tanh(z),    label="tanh")
plt.plot(z, relu(z),    label="ReLU")
plt.axhline(0, color="grey", lw=0.7)
plt.axvline(0, color="grey", lw=0.7)
plt.title("Three activation functions")
plt.xlabel("z"); plt.ylabel("activation(z)")
plt.legend(); plt.grid(True)
plt.show()

**Discussion.** Notice their ranges differ: sigmoid lives in $(0,1)$, tanh in
$(-1,1)$, ReLU in $[0,\infty)$. Both sigmoid and tanh **saturate** — for large
$|z|$ their slope is nearly $0$, which (we'll see in 13b) makes gradients tiny
and learning slow. ReLU has slope exactly $1$ for $z>0$, so it does not
saturate on the positive side; that is the main reason it dominates modern
deep networks. For our small 2-layer network in 13b, smooth sigmoid/tanh work
fine and make the calculus cleaner.

> **A whole notebook on this.** Activation functions are important enough that
> notebook **13c** is devoted to them: why nonlinearity is *required*, the
> derivatives of each, the **vanishing-gradient** problem, "dying ReLU", and a
> practical "which one to use when" guide. Come back to it after 13b.

### A neuron as a tiny *computation graph*

It helps to picture a neuron not as one formula but as a short **chain of simple
operations** — a *computation graph*:

$$
x \;\xrightarrow{\;\times w,\ +b\;}\; z \;\xrightarrow{\;\sigma\;}\; a.
$$

Read left to right, this is the **forward pass**: multiply by $w$, add $b$ to
get $z$, then squash to get $a$. Each arrow is one elementary step whose
derivative we know. In 13b we will run the *same* graph backwards — reusing the
values we computed on the way forward — to get gradients. That backward trip is
**backpropagation**. Keep this picture in mind: *a neural network is just a long
chain of simple operations, and calculus lets us differentiate the whole chain
one link at a time.*

## 4. A linearly separable dataset

Let's make a 2-class dataset in the plane that a single line *can* separate:
two Gaussian blobs, one labelled $0$ and one labelled $1$.

In [ ]:
n = 40                                   # points per class
# class 0: centred near (-2, -2);  class 1: centred near (2, 2)
X0 = rng.normal(loc=[-2, -2], scale=0.8, size=(n, 2))
X1 = rng.normal(loc=[ 2,  2], scale=0.8, size=(n, 2))

X = np.vstack([X0, X1])                   # shape (80, 2): all points
y = np.array([0]*n + [1]*n)              # shape (80,):  labels 0/1
print("X shape:", X.shape, " y shape:", y.shape)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(X0[:, 0], X0[:, 1], color="tab:red",  label="class 0")
plt.scatter(X1[:, 0], X1[:, 1], color="tab:blue", label="class 1")
plt.title("A linearly separable dataset")
plt.xlabel("x1"); plt.ylabel("x2")
plt.legend(); plt.grid(True); plt.axis("equal")
plt.show()

## 5. The perceptron learning rule

The classic perceptron (Rosenblatt, 1958) uses a **step** activation: predict
$1$ if $z = w\cdot x + b \ge 0$, else predict $0$. Training is a beautifully
simple loop. For each point $(x, y)$:

1. predict $\hat{y} = \text{step}(w\cdot x + b)$,
2. compute the error $e = y - \hat{y}$ (which is $-1$, $0$, or $+1$),
3. nudge the weights toward fixing that error:

$$
w \leftarrow w + \eta\, e\, x, \qquad b \leftarrow b + \eta\, e.
$$

Here $\eta$ is the **learning rate**. If a point is already correct, $e=0$ and
nothing changes. The famous *perceptron convergence theorem* guarantees this
finds a separating line in finite steps **if one exists**.

In [ ]:
def step(z):
    return 1 if z >= 0 else 0          # hard threshold

def train_perceptron(X, y, eta=0.1, epochs=20):
    w = np.zeros(2)                    # start from zero weights
    b = 0.0
    for _ in range(epochs):           # one epoch = one sweep over all points
        for xi, yi in zip(X, y):
            y_hat = step(np.dot(w, xi) + b)
            error = yi - y_hat        # -1, 0, or +1
            w = w + eta * error * xi  # the update rule
            b = b + eta * error
    return w, b

w_p, b_p = train_perceptron(X, y)
print("learned weights:", w_p, " bias:", b_p)

### Plotting the decision boundary

The boundary is the line where $z = 0$, i.e. $w_1 x_1 + w_2 x_2 + b = 0$.
Solving for $x_2$ gives $x_2 = -(w_1 x_1 + b)/w_2$.

In [ ]:
def plot_boundary_line(w, b, X0, X1, title):
    plt.figure(figsize=(5, 5))
    plt.scatter(X0[:, 0], X0[:, 1], color="tab:red",  label="class 0")
    plt.scatter(X1[:, 0], X1[:, 1], color="tab:blue", label="class 1")
    xs = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
    ys = -(w[0] * xs + b) / w[1]      # the line w.x + b = 0
    plt.plot(xs, ys, "k--", label="decision boundary")
    plt.title(title)
    plt.xlabel("x1"); plt.ylabel("x2")
    plt.legend(); plt.grid(True); plt.axis("equal")
    plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
    plt.show()

plot_boundary_line(w_p, b_p, X0, X1, "Perceptron decision boundary")

In [ ]:
# Accuracy: fraction of points predicted correctly
preds = np.array([step(np.dot(w_p, xi) + b_p) for xi in X])
print("training accuracy:", (preds == y).mean())   # should be 1.0

The dashed line cleanly separates the two blobs and the accuracy is $1.0$. For
**linearly separable** data, a single neuron is enough.

---
## ✍️ Exercise 1 (solution included)

The **derivative** of the sigmoid has a famously tidy form,
$\sigma'(z) = \sigma(z)\,(1-\sigma(z))$. We'll need it for backpropagation in
13b. Write `sigmoid_deriv(z)` and verify it against a numerical derivative
$\frac{\sigma(z+h)-\sigma(z-h)}{2h}$ at $z = 0.7$.

**Solution:**

In [ ]:
def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

z0 = 0.7
h = 1e-5
numeric = (sigmoid(z0 + h) - sigmoid(z0 - h)) / (2 * h)
print("formula:", sigmoid_deriv(z0))
print("numeric:", numeric)
print("difference:", abs(sigmoid_deriv(z0) - numeric))   # ~1e-11

## ✍️ Exercise 2 (solution included)

Run the perceptron with a **larger** learning rate `eta=1.0` and only
`epochs=5`. Does it still reach perfect accuracy on our separable blobs?
Plot the boundary.

**Solution:**

In [ ]:
w2, b2 = train_perceptron(X, y, eta=1.0, epochs=5)
preds2 = np.array([step(np.dot(w2, xi) + b2) for xi in X])
print("accuracy:", (preds2 == y).mean())
plot_boundary_line(w2, b2, X0, X1, "Perceptron (eta=1.0, 5 epochs)")
# Because the data is linearly separable, the perceptron converges quickly;
# a different eta just rescales w/b and gives a (possibly tilted) valid line.

---
## 6. Why a single neuron cannot solve XOR

Now the punchline that motivates the whole field. **XOR** ("exclusive or") is
the 4-point dataset

| $x_1$ | $x_2$ | label |
|------|------|-------|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

The two "1" points sit on one diagonal, the two "0" points on the other. **No
single straight line can separate them.** A single neuron only ever draws a
straight line ($w\cdot x + b = 0$), so it is *mathematically incapable* of
solving XOR. Let's watch it fail.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

w_x, b_x = train_perceptron(X_xor, y_xor, eta=0.1, epochs=100)
preds_x = np.array([step(np.dot(w_x, xi) + b_x) for xi in X_xor])
print("predictions:", preds_x)
print("targets    :", y_xor)
print("accuracy   :", (preds_x == y_xor).mean())   # stuck around 0.5

In [ ]:
# Visualize: there is NO straight line separating o from x
plt.figure(figsize=(5, 5))
for xi, yi in zip(X_xor, y_xor):
    marker = "x" if yi == 1 else "o"
    color  = "tab:blue" if yi == 1 else "tab:red"
    plt.scatter(xi[0], xi[1], marker=marker, color=color, s=200)
xs = np.linspace(-0.5, 1.5, 50)
ys = -(w_x[0] * xs + b_x) / (w_x[1] if w_x[1] != 0 else 1e-6)
plt.plot(xs, ys, "k--", label="best line the neuron found")
plt.title("XOR: no single line works")
plt.xlabel("x1"); plt.ylabel("x2")
plt.xlim(-0.5, 1.5); plt.ylim(-0.5, 1.5)
plt.legend(); plt.grid(True)
plt.show()

The accuracy is stuck near $0.5$ (no better than guessing) and the line cannot
possibly cut the two classes apart, no matter how long we train.

**The fix:** stack neurons in layers. A *hidden layer* first bends the space
(applying several lines + nonlinearities), and then the output neuron *can*
separate the transformed points. That curved decision boundary is exactly what
we build in **notebook 13b** — and the tool that trains it is **backpropagation**,
the chain rule from Chapter 07 applied layer by layer.

### A note before 13b: from a loop to *matrices* (vectorization)

Look again at `train_perceptron`: it has a Python `for` loop that handles **one
point at a time** (`for xi, yi in zip(X, y)`). That is easy to read, but slow,
and it hides the linear-algebra structure. In 13b we switch to **vectorization**:
instead of looping over the $N$ data points, we stack them as the rows of a
matrix $X$ (shape $N\times n$) and let a **single matrix multiply** $XW$ process
the whole batch at once.

This is not just a speed trick — it is how you should *think*. One matrix
product = "apply this layer to every example simultaneously." Andrew Ng's advice:
**whenever you see yourself writing a loop over training examples, ask whether a
matrix operation can replace it.** NumPy then runs that operation in fast
compiled code, and the math reads as clean linear algebra.

---
## 📝 Homework (no solutions provided)

1. **Leaky ReLU.** Implement $\text{leaky}(z) = z$ for $z>0$ and $0.01 z$
   otherwise, and add it to the activation plot. Why might a small negative
   slope help compared to plain ReLU?
2. **Track convergence.** Modify `train_perceptron` to record the number of
   misclassified points after each epoch, and plot that count versus epoch for
   the separable blobs. How many epochs until it hits zero?
3. **AND and OR are linearly separable.** Build the truth-table datasets for
   logical AND and OR and confirm a single perceptron learns each one
   perfectly (plot the boundaries). Contrast with XOR.
4. **Move the blobs together.** Reduce the gap between the two Gaussian
   centres until they overlap. At what point does the perceptron stop reaching
   100% accuracy, and why?